# Ultrason Görüntülerinden Kist Tespiti

Bu notebook, göğüs ultrason görüntülerinde kist tespiti yapmak için derin öğrenme tabanlı bir görüntü işleme projesidir.

## 📋 İçindekiler
1. **ADIM 1: Model Eğitimi** - Veri yükleme, model oluşturma ve eğitimi
2. **ADIM 2: GUI Uygulaması** - Kullanıcı dostu arayüz ile tahmin yapma


# ADIM 1: Model Eğitimi

Bu hücrede veriler yüklenecek, model oluşturulacak ve eğitilecektir.


In [ ]:
# ============================================================================
# ADIM 1: ULTRASON KİST TESPİTİ (FİNAL VERSİYON - İNCE AYAR EKLENDİ)
# ============================================================================

import os
import numpy as np
import cv2
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report
import tensorflow as tf 
import warnings
warnings.filterwarnings('ignore')

# --- TEMEL AYARLAR ---
IMAGE_WIDTH, IMAGE_HEIGHT = 224, 224
BATCH_SIZE = 32
INITIAL_EPOCHS = 10 # İlk aşama için daha kısa
FINE_TUNE_EPOCHS = 15 # İkinci aşama için
MODEL_SAVE_PATH = 'models/best_model_ft.h5' # İnce ayar modelini farklı kaydet
INITIAL_LR = 0.00002 # Ön eğitim öğrenme hızı
FINE_TUNE_LR = 0.000001 # İnce ayar için daha da düşük öğrenme hızı

# --- VERİ YÜKLEME VE ÖN İŞLEME ---
def load_images_from_folder(folder_path, label):
    images, labels = [], []
    for root, _, files in os.walk(folder_path):
        for filename in files:
            if filename.lower().endswith(('.jpg', '.png')):
                img_path = os.path.join(root, filename)
                try:
                    img = cv2.imread(img_path)
                    if img is None: img = np.array(Image.open(img_path))
                    if len(img.shape) == 2: img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
                    if len(img.shape) == 3 and img.shape[2] == 4: img = cv2.cvtColor(img, cv2.COLOR_RGBA2RGB)
                    if len(img.shape) == 3 and img.shape[2] == 3: img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    
                    img = cv2.resize(img, (IMAGE_WIDTH, IMAGE_HEIGHT)).astype(np.float32) / 255.0
                    images.append(img); labels.append(label)
                except Exception as e:
                    pass
    return np.array(images), np.array(labels)

# Veri yükleme ve Bölme
healthy, h_labels = load_images_from_folder(os.path.join('data', 'healthy'), 0)
cystic, c_labels = load_images_from_folder(os.path.join('data', 'cystic'), 1)
all_images = np.concatenate([healthy, cystic], axis=0)
all_labels = np.concatenate([h_labels, c_labels], axis=0)

X_train, X_test, y_train, y_test = train_test_split(all_images, all_labels, test_size=0.2, random_state=42, stratify=all_labels)
X_train, X_val, y_train_raw, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=42, stratify=y_train)

# Sınıf Ağırlıkları ve Encoding
class_weights = compute_class_weight('balanced', classes=np.unique(y_train_raw), y=y_train_raw)
class_weight_dict = {i: class_weights[i] for i in range(len(class_weights))}
y_train = tf.keras.utils.to_categorical(y_train_raw, 2)
y_val = tf.keras.utils.to_categorical(y_val, 2)
y_test = tf.keras.utils.to_categorical(y_test, 2)

print(f"\n📈 Veri Dağılımı: Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
print(f"⚖️ Sınıf Ağırlıkları: {class_weight_dict}")

# --- MODEL OLUŞTURMA (TRANSFER LEARNING) ---
def create_base_model(input_shape, lr):
    # Tüm katmanlar dondurulmuş MobileNetV2
    base_model = tf.keras.applications.MobileNetV2(input_shape=input_shape, include_top=False, weights='imagenet')
    for layer in base_model.layers: layer.trainable = False 

    x = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
    x = tf.keras.layers.Dense(256, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001))(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    predictions = tf.keras.layers.Dense(2, activation='softmax')(x)

    model = tf.keras.models.Model(inputs=base_model.input, outputs=predictions)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr), loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# --- EĞİTİM İÇİN HAZIRLIK ---
model = create_base_model(input_shape=(IMAGE_WIDTH, IMAGE_HEIGHT, 3), lr=INITIAL_LR) 
os.makedirs('models', exist_ok=True)

# Geri Çağrılar
initial_checkpoint = tf.keras.callbacks.ModelCheckpoint('models/best_model_initial.h5', monitor='val_accuracy', save_best_only=True, mode='max', verbose=1)
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1, min_delta=0.0001, mode='min')
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=5, min_lr=0.000001, verbose=1)

# Data Augmentation
datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rotation_range=30, width_shift_range=0.15, height_shift_range=0.15,
    shear_range=0.15, zoom_range=0.2, horizontal_flip=True, vertical_flip=True,
    brightness_range=[0.8, 1.2], fill_mode='nearest'
)

# --- 1. AŞAMA: ÖN EĞİTİM (DONDURULMUŞ KATMANLARLA) ---
print("\n" + "="*70); print("1. AŞAMA: ÖN EĞİTİM BAŞLIYOR (Yeni Katmanlar Eğitiliyor)"); print("="*70)

model.fit(
    datagen.flow(X_train, y_train, batch_size=BATCH_SIZE),
    steps_per_epoch=len(X_train) // BATCH_SIZE,
    epochs=INITIAL_EPOCHS,
    validation_data=(X_val, y_val),
    callbacks=[initial_checkpoint, early_stopping, reduce_lr],
    class_weight=class_weight_dict,
    verbose=1 
)
# --- 2. AŞAMA: İNCE AYAR (FINE-TUNING) ---

# En iyi ön eğitim modelini yükle
model = tf.keras.models.load_model('models/best_model_initial.h5', compile=False)

# MobileNetV2'nin son 50 katmanını eğitime aç (Tıbbi veriye adaptasyon)
for layer in model.layers:
    if layer.name == 'global_average_pooling2d': # MobilnetV2'nin son katmanına kadar dondur
        break
    layer.trainable = True # Son katmanları eğitime aç

# İnce ayar için DAHA DÜŞÜK öğrenme hızı ile derle
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=FINE_TUNE_LR), 
    loss='categorical_crossentropy', 
    metrics=['accuracy']
)

# İnce Ayar için yeni geri çağrılar (val_loss'u izlemeye devam et)
ft_checkpoint = tf.keras.callbacks.ModelCheckpoint(MODEL_SAVE_PATH, monitor='val_accuracy', save_best_only=True, mode='max', verbose=1)

print("\n" + "="*70); print("2. AŞAMA: İNCE AYAR BAŞLIYOR (Model Uyarlanıyor)"); print("="*70)

model.fit(
    datagen.flow(X_train, y_train, batch_size=BATCH_SIZE),
    steps_per_epoch=len(X_train) // BATCH_SIZE,
    epochs=INITIAL_EPOCHS + FINE_TUNE_EPOCHS, # Toplam epoch sayısını artır
    initial_epoch=INITIAL_EPOCHS, # Başlangıç epoch'u 1. aşamanın sonu olarak ayarla
    validation_data=(X_val, y_val),
    callbacks=[ft_checkpoint, early_stopping, reduce_lr],
    class_weight=class_weight_dict,
    verbose=1
)

print("="*70); print("EĞİTİM TAMAMLANDI"); print("="*70)

# --- DEĞERLENDİRME ---
best_model = tf.keras.models.load_model(MODEL_SAVE_PATH, compile=False)

# Değerlendirmeden önce modeli derle (Hata giderme adımı)
best_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=FINE_TUNE_LR), 
    loss='categorical_crossentropy', 
    metrics=['accuracy']
)

test_loss, test_accuracy = best_model.evaluate(X_test, y_test, verbose=0)
print(f"\n📊 Test Accuracy: {test_accuracy:.4f}")
print(f"📊 Test Loss: {test_loss:.4f}")

y_pred = best_model.predict(X_test, verbose=0)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

print("\n📋 Sınıflandırma Raporu:")
print(classification_report(y_true_classes, y_pred_classes, target_names=['Sağlıklı', 'Kistli']))


📈 Veri Dağılımı: Train: 605, Val: 68, Test: 169
⚖️ Sınıf Ağırlıkları: {0: np.float64(1.0016556291390728), 1: np.float64(0.9983498349834984)}

MODEL EĞİTİMİ BAŞLIYOR

MODEL EĞİTİMİ BAŞLIYOR
Epoch 1/25
Epoch 1/25
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 896ms/step - accuracy: 0.4327 - loss: 1.7779
Epoch 1: val_accuracy improved from None to 0.50000, saving model to models/best_model.h5

Epoch 1: val_accuracy improved from None to 0.50000, saving model to models/best_model.h5


18/18 ━━━━━━━━━━━━━━━━━━━━ 29s 1s/step - accuracy: 0.4520 - loss: 1.6119 - val_accuracy: 0.5000 - val_loss: 1.4804 - learning_rate: 2.0000e-05
Epoch 2/25
Epoch 2/25
 1/18 ━━━━━━━━━━━━━━━━━━━━ 11s 697ms/step - accuracy: 0.8125 - loss: 1.1266
Epoch 2: val_accuracy did not improve from 0.50000

Epoch 2: val_accuracy did not improve from 0.50000
18/18 ━━━━━━━━━━━━━━━━━━━━ 2s 94ms/step - accuracy: 0.8125 - loss: 1.1266 - val_accuracy: 0.5000 - val_loss: 1.4707 - learning_rate: 2.0000e-05
Epoch 3/25
18/18 ━━━━━━━━━━━━━━━━━━━━ 2s 94ms/step - accuracy: 0.8125 - loss: 1.1266 - val_accuracy: 0.5000 - val_loss: 1.4707 - learning_rate: 2.0000e-05
Epoch 3/25
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 972ms/step - accuracy: 0.4820 - loss: 1.3768
Epoch 3: val_accuracy improved from 0.50000 to 0.51471, saving model to models/best_model.h5

Epoch 3: val_accuracy improved from 0.50000 to 0.51471, saving model to models/best_model.h5


18/18 ━━━━━━━━━━━━━━━━━━━━ 20s 1s/step - accuracy: 0.4991 - loss: 1.2878 - val_accuracy: 0.5147 - val_loss: 1.3236 - learning_rate: 2.0000e-05
Epoch 4/25
Epoch 4/25
 1/18 ━━━━━━━━━━━━━━━━━━━━ 12s 707ms/step - accuracy: 0.4688 - loss: 1.2154
Epoch 4: val_accuracy did not improve from 0.51471

Epoch 4: val_accuracy did not improve from 0.51471
18/18 ━━━━━━━━━━━━━━━━━━━━ 2s 98ms/step - accuracy: 0.4688 - loss: 1.2154 - val_accuracy: 0.5147 - val_loss: 1.3156 - learning_rate: 2.0000e-05
Epoch 5/25
18/18 ━━━━━━━━━━━━━━━━━━━━ 2s 98ms/step - accuracy: 0.4688 - loss: 1.2154 - val_accuracy: 0.5147 - val_loss: 1.3156 - learning_rate: 2.0000e-05
Epoch 5/25
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 900ms/step - accuracy: 0.5742 - loss: 1.1681
Epoch 5: val_accuracy improved from 0.51471 to 0.55882, saving model to models/best_model.h5

Epoch 5: val_accuracy improved from 0.51471 to 0.55882, saving model to models/best_model.h5


18/18 ━━━━━━━━━━━━━━━━━━━━ 18s 1s/step - accuracy: 0.5672 - loss: 1.1690 - val_accuracy: 0.5588 - val_loss: 1.1702 - learning_rate: 2.0000e-05
Epoch 6/25
Epoch 6/25
 1/18 ━━━━━━━━━━━━━━━━━━━━ 11s 680ms/step - accuracy: 0.4688 - loss: 1.5374
Epoch 6: val_accuracy improved from 0.55882 to 0.57353, saving model to models/best_model.h5

Epoch 6: val_accuracy improved from 0.55882 to 0.57353, saving model to models/best_model.h5


18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 111ms/step - accuracy: 0.4688 - loss: 1.5374 - val_accuracy: 0.5735 - val_loss: 1.1654 - learning_rate: 2.0000e-05
Epoch 7/25
Epoch 7/25
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 902ms/step - accuracy: 0.5689 - loss: 1.2154
Epoch 7: val_accuracy improved from 0.57353 to 0.64706, saving model to models/best_model.h5

Epoch 7: val_accuracy improved from 0.57353 to 0.64706, saving model to models/best_model.h5


18/18 ━━━━━━━━━━━━━━━━━━━━ 19s 1s/step - accuracy: 0.5986 - loss: 1.1840 - val_accuracy: 0.6471 - val_loss: 1.0920 - learning_rate: 2.0000e-05
Epoch 8/25
Epoch 8/25
 1/18 ━━━━━━━━━━━━━━━━━━━━ 13s 766ms/step - accuracy: 0.5625 - loss: 1.0810
Epoch 8: val_accuracy did not improve from 0.64706

Epoch 8: val_accuracy did not improve from 0.64706
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 106ms/step - accuracy: 0.5625 - loss: 1.0810 - val_accuracy: 0.6471 - val_loss: 1.0892 - learning_rate: 2.0000e-05
Epoch 9/25
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 106ms/step - accuracy: 0.5625 - loss: 1.0810 - val_accuracy: 0.6471 - val_loss: 1.0892 - learning_rate: 2.0000e-05
Epoch 9/25
15/18 ━━━━━━━━━━━━━━━━━━━━ 3s 1s/step - accuracy: 0.5844 - loss: 1.1583

KeyboardInterrupt: 

# ADIM 2: GUI Uygulaması

Bu hücrede eğitilmiş modeli kullanarak görsel bir arayüz ile tahmin yapabilirsiniz.


In [10]:
# ============================================================================
# GUI UYGULAMASI - Ultrason Kist Tespiti
# ============================================================================

import tkinter as tk
from tkinter import filedialog, Label, Frame
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
import os
from PIL import Image, ImageTk

# --- UYGULAMA AYARLARI ---
MODEL_PATH = 'models/best_model.h5'
IMAGE_WIDTH, IMAGE_HEIGHT = 224, 224
WINDOW_WIDTH, WINDOW_HEIGHT = 800, 650

# Global değişkenler
cap = None
is_camera_running = False
current_frame = None

# --- 1. MODELİ VE SINIFLARI YÜKLEME ---
if not os.path.exists(MODEL_PATH):
    print(f"HATA: Model dosyası '{MODEL_PATH}' bulunamadı.")
    print("Lütfen önce eğitim hücresini çalıştırarak modeli eğitin.")
else:
    # DÜZELTME 1: Modeli sadece tahmin için derlemeden yükle (Uyarıyı kaldırır)
    model = load_model(MODEL_PATH, compile=False)
    class_names = ['Sağlıklı', 'Kistli']
    print("Model ve sınıflar başarıyla yüklendi.")

# --- 2. GÜVENLİ GÖRÜNTÜ OKUMA VE TAHMİN FONKSİYONLARI ---
def imread_utf8(filepath):
    try:
        with open(filepath, 'rb') as f:
            stream = f.read()
        return cv2.imdecode(np.frombuffer(stream, np.uint8), cv2.IMREAD_COLOR)
    except Exception as e:
        print(f"Görüntü okuma sırasında kritik hata: {e}")
        return None

def predict_on_image(image):
    image_for_prediction = cv2.resize(image, (IMAGE_WIDTH, IMAGE_HEIGHT))
    image_for_prediction = cv2.cvtColor(image_for_prediction, cv2.COLOR_BGR2RGB)
    image_for_prediction = image_for_prediction.astype("float") / 255.0
    image_for_prediction = np.expand_dims(image_for_prediction, axis=0)
    
    # DÜZELTME 2: Tahmin işlemini sessiz modda yap (Tekrarlayan çıktıları kaldırır)
    predictions = model.predict(image_for_prediction, verbose=0)[0]
    
    prediction_index = np.argmax(predictions)
    confidence = predictions[prediction_index] * 100
    predicted_class = class_names[prediction_index]
    
    return f"Tahmin: {predicted_class} ({confidence:.2f}%)"

def update_camera_feed():
    global is_camera_running, current_frame
    if not is_camera_running: return
    ret, frame = cap.read()
    if ret:
        current_frame = frame.copy()
        label = predict_on_image(frame)
        prediction_label.config(text=label)
        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        img_pil = Image.fromarray(img_rgb)
        img_tk = ImageTk.PhotoImage(image=img_pil)
        
        image_label.imgtk = img_tk
        image_label.config(image=img_tk)
        image_label.after(15, update_camera_feed)
    else:
        stop_camera()

def start_live_camera():
    global cap, is_camera_running
    if is_camera_running: return
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        prediction_label.config(text="HATA: Kamera başlatılamadı.")
        return
    
    is_camera_running = True
    capture_button.config(state=tk.NORMAL)
    update_camera_feed()

def stop_camera():
    global cap, is_camera_running
    if cap:
        cap.release()
    is_camera_running = False
    capture_button.config(state=tk.DISABLED)
    prediction_label.config(text="Kamera durduruldu.")

def capture_and_save():
    global is_camera_running
    if not is_camera_running or current_frame is None: return
    
    is_camera_running = False
    capture_button.config(state=tk.DISABLED)
    prediction_label.config(text="Görüntü yakalandı. Kaydetmek için konum seçin.")
    file_path = filedialog.asksaveasfilename(
        defaultextension=".jpg",
        filetypes=[("JPEG files", "*.jpg"), ("PNG files", "*.png")],
        title="Yakalanan Görüntüyü Kaydet"
    )
    
    if file_path:
        cv2.imwrite(file_path, current_frame)
        prediction_label.config(text=f"Görüntü şuraya kaydedildi:\n{os.path.basename(file_path)}")
    else:
        prediction_label.config(text="Kaydetme işlemi iptal edildi.")

def load_and_predict_image():
    stop_camera()
    file_path = filedialog.askopenfilename(
        title="Bir Görüntü Dosyası Seçin",
        filetypes=[("Image Files", "*.jpg *.jpeg *.png")]
    )
    if not file_path: return
    original_image = imread_utf8(file_path)
    
    if original_image is None:
        prediction_label.config(text=f"HATA: Görüntü okunamadı!\n{os.path.basename(file_path)}")
        return
    label = predict_on_image(original_image)
    prediction_label.config(text=label)
    img_rgb = cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB)
    h, w, _ = img_rgb.shape
    scale = min((WINDOW_WIDTH-50)/w, (WINDOW_HEIGHT-150)/h)
    new_w, new_h = int(w*scale), int(h*scale)
    
    img_pil = Image.fromarray(img_rgb).resize((new_w, new_h))
    img_tk = ImageTk.PhotoImage(image=img_pil)
    
    image_label.imgtk = img_tk
    image_label.config(image=img_tk)

# --- 3. ANA ARAYÜZ (GUI) PENCERESİNİ OLUŞTURMA ---
if 'model' in globals() and model is not None:
    root = tk.Tk()
    root.title("Ultrason Kist Tespiti Uygulaması")
    root.geometry(f"{WINDOW_WIDTH}x{WINDOW_HEIGHT}")
    root.configure(bg="#2E2E2E")
    
    top_frame = Frame(root, bg="#2E2E2E")
    top_frame.pack(pady=10)
    
    camera_button = tk.Button(top_frame, text="Canlı Kamera Başlat", command=start_live_camera, font=("Helvetica", 12), width=18, bg="#4CAF50", fg="white")
    camera_button.pack(side=tk.LEFT, padx=5)
    
    capture_button = tk.Button(top_frame, text="Fotoğraf Çek", command=capture_and_save, font=("Helvetica", 12), width=15, bg="#FF9800", fg="white", state=tk.DISABLED)
    capture_button.pack(side=tk.LEFT, padx=5)
    
    file_button = tk.Button(top_frame, text="Dosyadan Görüntü Yükle", command=load_and_predict_image, font=("Helvetica", 12), width=20, bg="#008CBA", fg="white")
    file_button.pack(side=tk.LEFT, padx=5)
    
    image_label = Label(root, bg="#1C1C1C")
    image_label.pack(pady=10, padx=20, expand=True, fill="both")
    
    prediction_label = Label(root, text="Lütfen bir işlem seçin.", font=("Helvetica", 14, "bold"), bg="#2E2E2E", fg="white")
    prediction_label.pack(pady=10)
    
    def on_closing():
        stop_camera()
        root.destroy()
    
    root.protocol("WM_DELETE_WINDOW", on_closing)
    root.mainloop()
else:
    print("⚠ Model yüklenemediği için GUI başlatılamadı.")
    print("   Lütfen önce ADIM 1'i çalıştırarak modeli eğitin.")


Model ve sınıflar başarıyla yüklendi.
